## Предобработка данных для дальнейшего анализа

Цель: подготовить, очистить данные для дальнейшего анализа и построения дашборда

План работ:
- выгрузка данных из БД/csv (csv - как резервный источник в случае недоступности БД)
- предобработка, очистка 
- загрузка очищенных данных в БД/csv

In [2]:
# импорт библиотек
import pandas as pd
import numpy as np

import time
from datetime import datetime, timedelta
from typing import List, Dict, Optional

import sqlalchemy
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os


### 1. Выгрузка и предобработка данных

**Подключение к БД**

In [3]:
# Подключение к БД Supabase
load_dotenv()

# Параметры подключения
user = os.getenv('user')
password = os.getenv('password')
host = os.getenv('host')
port = '6543'
dbname = os.getenv('dbname')

# Адрес подключения
DATABASE_URL = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}?sslmode=require"

# Формируем движок
engine = create_engine(DATABASE_URL) 

try:
    with engine.connect() as connection:
        print(f'Подключение к базе успешно')
except Exception as e:
        print(f'Ошибка при подключении к базе: {e}')       
    
   

Ошибка при подключении к базе: (psycopg2.OperationalError) server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

[SQL: select pg_catalog.version()]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


#### 1.1 основная информация о ДТП

In [4]:
try:
    query_main = '''
           SELECT *
           FROM main_df;
          '''
    main_df = pd.read_sql_query(query_main, con=engine)

except: 
    main_df = pd.read_csv('main_df.csv')

In [5]:
main_df.info()
main_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20224 entries, 0 to 20223
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   index        20224 non-null  int64  
 1   KartId       20224 non-null  int64  
 2   rowNum       20224 non-null  int64  
 3   date         20224 non-null  object 
 4   Time         20224 non-null  object 
 5   District     20224 non-null  object 
 6   DTP_V        20224 non-null  object 
 7   POG          20224 non-null  int64  
 8   RAN          20224 non-null  int64  
 9   K_TS         20224 non-null  int64  
 10  K_UCH        20224 non-null  int64  
 11  emtp_number  10962 non-null  float64
dtypes: float64(1), int64(7), object(4)
memory usage: 1.9+ MB


,index,KartId,rowNum,date,Time,District,DTP_V,POG,RAN,K_TS,K_UCH,emtp_number
0,75,157114292,76,02.01.2015,12:30,Верх-Исетский,Наезд на стоящее ТС,0,1,2,3,NaN
1,76,157122648,77,02.01.2015,17:15,Кировский,Наезд на пешехода,0,2,1,3,NaN
2,81,157156173,82,01.01.2015,22:00,Октябрьский,Наезд на препятствие,0,1,1,1,NaN
3,177,157223255,95,01.01.2015,02:20,Первомайский,Столкновение,0,2,2,3,NaN
4,178,157225233,96,01.01.2015,14:30,Первореченский,Наезд на пешехода,0,1,1,2,NaN


In [6]:
def clean_column_names(df):
    '''
    Убирает пробелы в начале и конце, заменяет внутренние пробелы на _
    '''
   
    df.columns = df.columns.str.strip()
    df.columns = df.columns.str.replace(' ', '_')
    return df

In [7]:
main_clean = main_df.drop(columns = ['index','rowNum','emtp_number']) # удалим лишние столбцы

# мнеяем формат даты
main_clean['date'] = pd.to_datetime(main_clean['date'], format='%d.%m.%Y')
# main['Time'] = pd.to_datetime(main['Time'], format='%H:%M').dt.time

# добавляем отдельные столбы с годов, месяцем, днем
main_clean['year'] = main_clean['date'].dt.year
main_clean['month'] = main_clean['date'].dt.month
main_clean['day'] = main_clean['date'].dt.day

#переименуем столбцы
main_clean = main_clean.rename(columns={
                'District':'district',
                'DTP_V': 'dtp_type',
                'POG': 'dead', 
                'RAN': 'injured',
                'K_TS': 'ts_quantity',
                'K_UCH': 'people_quantity',
                'Time':'time', 
                })
clean_column_names(main_clean)
main_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20224 entries, 0 to 20223
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   KartId           20224 non-null  int64         
 1   date             20224 non-null  datetime64[ns]
 2   time             20224 non-null  object        
 3   district         20224 non-null  object        
 4   dtp_type         20224 non-null  object        
 5   dead             20224 non-null  int64         
 6   injured          20224 non-null  int64         
 7   ts_quantity      20224 non-null  int64         
 8   people_quantity  20224 non-null  int64         
 9   year             20224 non-null  int32         
 10  month            20224 non-null  int32         
 11  day              20224 non-null  int32         
dtypes: datetime64[ns](1), int32(3), int64(5), object(3)
memory usage: 1.6+ MB


In [8]:
# проверка уникальности карточки с номером
main_clean['KartId'].duplicated().sum()

np.int64(0)

In [9]:
main_clean['district'].unique()

array(['Верх-Исетский', 'Кировский', 'Октябрьский', 'Первомайский',
       'Первореченский', 'Советский', 'Ленинский', 'Орджоникидзевский',
       'Ленинский Екб', 'Фрунзенский', 'Железнодорожный', 'Чкаловский',
       'Академический'], dtype=object)

In [10]:
main_clean['dtp_type'].value_counts()

dtp_type
Столкновение                                                                                                 7856
Наезд на пешехода                                                                                            7769
Наезд на препятствие                                                                                         1306
Падение пассажира                                                                                            1217
Наезд на стоящее ТС                                                                                           649
Наезд на велосипедиста                                                                                        569
Съезд с дороги                                                                                                354
Опрокидывание                                                                                                 298
Иной вид ДТП                                                                   

**Создадим календарь, выделим дату и KartID в вспомогательную отдельную таблицу**

In [11]:
calendar = main_clean[['KartId','date','time']]
calendar

,KartId,date,time
0,157114292,2015-01-02,12:30
1,157122648,2015-01-02,17:15
2,157156173,2015-01-01,22:00
3,157223255,2015-01-01,02:20
4,157225233,2015-01-01,14:30
...,...,...,...
20219,225394938,2025-12-20,15:20
20220,225419708,2025-09-11,12:21
20221,225419832,2025-11-12,10:45
20222,225420254,2025-12-23,11:00


In [12]:
calendar['KartId'].duplicated().sum()

np.int64(0)

#### 1.2 Информация о дорогах и погодных условиях

In [13]:
try:
    query_info = '''
           SELECT *
           FROM info_dtp_df;
          '''
    info_df = pd.read_sql_query(query_info, con=engine)

except: 
    info_df = pd.read_csv('info_dtp_df.csv')

In [14]:
info_df.info()
info_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20224 entries, 0 to 20223
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   index              20224 non-null  int64  
 1   KartId             20224 non-null  int64  
 2   n_p                20187 non-null  object 
 3   street             17200 non-null  object 
 4   house              17196 non-null  object 
 5   dor                3022 non-null   object 
 6   km                 20221 non-null  float64
 7   m                  20222 non-null  float64
 8   k_ul               19385 non-null  object 
 9   dor_k              2104 non-null   float64
 10  dor_z              20224 non-null  object 
 11  s_pch              20224 non-null  object 
 12  osv                20224 non-null  object 
 13  change_org_motion  20224 non-null  object 
 14  s_dtp              20224 non-null  int64  
 15  COORD_W            20224 non-null  float64
 16  COORD_L            202

,index,KartId,n_p,street,house,dor,km,m,k_ul,dor_k,...,osv,change_org_motion,s_dtp,COORD_W,COORD_L,ndu,sdor,factor,s_pog,OBJ_DTP
0,0,161237026,г Екатеринбург,ул Бажова,99,NaN,0.0,0.0,Улицы и дороги местного значения в жилой застр...,NaN,...,Светлое время суток,Режим движения не изменялся,910,56.838333,60.628056,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),Сведения отсутствуют,Ясно,Многоквартирные жилые дома
1,1,161340310,г Екатеринбург,ул Академика Шварца,17,NaN,0.0,0.0,Улицы и дороги местного значения в жилой застр...,NaN,...,Светлое время суток,Режим движения не изменялся,880,56.796667,60.626667,Не установлены,"Перегон (нет объектов на месте ДТП), Автостоян...",Сведения отсутствуют,Ясно,"Остановка общественного транспорта, Крупный то..."
2,2,161238027,г Екатеринбург,ул Репина,6,NaN,0.0,0.0,Магистральные улицы районного значения,NaN,...,"В темное время суток, освещение включено",Режим движения не изменялся,940,56.830278,60.573889,"Отсутствие, плохая различимость горизонтальной...",Перегон (нет объектов на месте ДТП),Сведения отсутствуют,Пасмурно,"Лечебные учреждения, Спортивные и развлекатель..."
3,3,171517613,г Екатеринбург,ул Софьи Ковалевской,1,NaN,0.0,0.0,Магистральные улицы районного значения,NaN,...,"В темное время суток, освещение включено",Режим движения не изменялся,710,56.833056,60.600000,Нарушения в размещении наружной рекламы,"Регулируемый пешеходный переход, Регулируемый ...",Сведения отсутствуют,Ясно,"Остановка общественного транспорта, Остановка ..."
4,4,160935023,г Екатеринбург,ул Малышева,6,NaN,0.0,0.0,Магистральные улицы районного значения,NaN,...,"В темное время суток, освещение включено",Движение частично перекрыто,500,56.831667,60.583056,"Отсутствие, плохая различимость горизонтальной...","Перегон (нет объектов на месте ДТП), Регулируе...",Сведения отсутствуют,Пасмурно,"Остановка общественного транспорта, Остановка ..."


In [15]:
info_dtp_clean = info_df.drop(columns = ['index','street','dor',
                                        'house','km','m','COORD_W','COORD_L', 's_dtp', 'OBJ_DTP','dor_k']) # удалим лишние столбцы

#переименуем столбцы
info_dtp_clean = info_dtp_clean.rename(columns={
                'k_ul': 'street_category', 'dor_z': 'road_category',
                            's_pch': 'road_coverage', 'osv': 'light',
                            'ndu': 'road_problems', 'sdor': 'scheme',
                             's_pog': 'weather','n_p':'city'
                            })
clean_column_names(info_dtp_clean)

,KartId,city,street_category,road_category,road_coverage,light,change_org_motion,road_problems,scheme,factor,weather
0,161237026,г Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Режим движения не изменялся,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),Сведения отсутствуют,Ясно
1,161340310,г Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Режим движения не изменялся,Не установлены,"Перегон (нет объектов на месте ДТП), Автостоян...",Сведения отсутствуют,Ясно
2,161238027,г Екатеринбург,Магистральные улицы районного значения,Не указано,Обработанное противогололедными материалами,"В темное время суток, освещение включено",Режим движения не изменялся,"Отсутствие, плохая различимость горизонтальной...",Перегон (нет объектов на месте ДТП),Сведения отсутствуют,Пасмурно
3,171517613,г Екатеринбург,Магистральные улицы районного значения,Не указано,Гололедица,"В темное время суток, освещение включено",Режим движения не изменялся,Нарушения в размещении наружной рекламы,"Регулируемый пешеходный переход, Регулируемый ...",Сведения отсутствуют,Ясно
4,160935023,г Екатеринбург,Магистральные улицы районного значения,Не указано,Сухое,"В темное время суток, освещение включено",Движение частично перекрыто,"Отсутствие, плохая различимость горизонтальной...","Перегон (нет объектов на месте ДТП), Регулируе...",Сведения отсутствуют,Пасмурно
...,...,...,...,...,...,...,...,...,...,...,...
20219,225219802,г Владивосток,Улицы и дороги местного значения научно-произв...,"Местного значения (дорога местного значения, в...",Со снежным накатом,Светлое время суток,Движение частично перекрыто,Недостатки зимнего содержания,Нерегулируемый перекрёсток равнозначных улиц (...,Сведения отсутствуют,Ясно
20220,225222817,г Владивосток,Магистральные улицы общегородского значения,"Местного значения (дорога местного значения, в...",Мокрое,Светлое время суток,Режим движения не изменялся,Не установлены,Нерегулируемый перекрёсток неравнозначных улиц...,Сведения отсутствуют,Ясно
20221,225228096,г Владивосток,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",Мокрое,"В темное время суток, освещение включено",Движение частично перекрыто,Не установлены,Перегон (нет объектов на месте ДТП),Сведения отсутствуют,Ясно
20222,225219810,г Владивосток,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",Мокрое,Светлое время суток,Движение частично перекрыто,Не установлены,Нерегулируемый перекрёсток неравнозначных улиц...,Сведения отсутствуют,Ясно


In [16]:
info_dtp_clean.info()
info_dtp_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20224 entries, 0 to 20223
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   KartId             20224 non-null  int64 
 1   city               20187 non-null  object
 2   street_category    19385 non-null  object
 3   road_category      20224 non-null  object
 4   road_coverage      20224 non-null  object
 5   light              20224 non-null  object
 6   change_org_motion  20224 non-null  object
 7   road_problems      20224 non-null  object
 8   scheme             20224 non-null  object
 9   factor             20224 non-null  object
 10  weather            20224 non-null  object
dtypes: int64(1), object(10)
memory usage: 1.7+ MB


,KartId,city,street_category,road_category,road_coverage,light,change_org_motion,road_problems,scheme,factor,weather
0,161237026,г Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Режим движения не изменялся,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),Сведения отсутствуют,Ясно
1,161340310,г Екатеринбург,Улицы и дороги местного значения в жилой застр...,Не указано,Сухое,Светлое время суток,Режим движения не изменялся,Не установлены,"Перегон (нет объектов на месте ДТП), Автостоян...",Сведения отсутствуют,Ясно
2,161238027,г Екатеринбург,Магистральные улицы районного значения,Не указано,Обработанное противогололедными материалами,"В темное время суток, освещение включено",Режим движения не изменялся,"Отсутствие, плохая различимость горизонтальной...",Перегон (нет объектов на месте ДТП),Сведения отсутствуют,Пасмурно
3,171517613,г Екатеринбург,Магистральные улицы районного значения,Не указано,Гололедица,"В темное время суток, освещение включено",Режим движения не изменялся,Нарушения в размещении наружной рекламы,"Регулируемый пешеходный переход, Регулируемый ...",Сведения отсутствуют,Ясно
4,160935023,г Екатеринбург,Магистральные улицы районного значения,Не указано,Сухое,"В темное время суток, освещение включено",Движение частично перекрыто,"Отсутствие, плохая различимость горизонтальной...","Перегон (нет объектов на месте ДТП), Регулируе...",Сведения отсутствуют,Пасмурно


In [17]:
# проверка уникальности карточки с номером
info_dtp_clean['KartId'].duplicated().sum()

np.int64(0)

In [18]:
info_dtp_clean['city'].value_counts()

city
г Екатеринбург           10688
г Владивосток             8939
остров Русский             228
п Трудовое                 162
с Горный Щит                53
п Русский                   24
п Шабровский                21
п Садовый                   13
п Полеводство               12
п Северка                   10
п Шувакиш                    6
с Береговое                  6
п Сысерть                    5
п Исток                      3
остров Попова                3
п Палкинский Торфяник        3
п Ягодный                    2
п Совхозный                  2
с Верхнемакарово             2
п Зеленый Бор                2
п Медный                     1
остров Рейнеке               1
п Глубокое                   1
Name: count, dtype: int64

In [19]:
# почистим названия городов, убрав лишние символы
info_dtp_clean['city'] = info_dtp_clean['city'].str.replace(r'^г\.?\s*', '', regex=True)

In [20]:
# оставим для дальнейшего анализа только Екатеринбург и Владивосток,
# т.к. остальные города-спутники и поселки существенной роли не играют
info_dtp_clean = info_dtp_clean.loc[info_dtp_clean['city'].isin(['Екатеринбург', 'Владивосток'])]

In [21]:
info_dtp_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19627 entries, 0 to 20223
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   KartId             19627 non-null  int64 
 1   city               19627 non-null  object
 2   street_category    18808 non-null  object
 3   road_category      19627 non-null  object
 4   road_coverage      19627 non-null  object
 5   light              19627 non-null  object
 6   change_org_motion  19627 non-null  object
 7   road_problems      19627 non-null  object
 8   scheme             19627 non-null  object
 9   factor             19627 non-null  object
 10  weather            19627 non-null  object
dtypes: int64(1), object(10)
memory usage: 1.8+ MB


In [22]:
info_dtp_clean['city'].unique()

array(['Екатеринбург', 'Владивосток'], dtype=object)

In [24]:
cities_coords = pd.read_csv('cities_coords.csv')


**выделим город и KartId в вспомогательную отдельную таблицу**

In [25]:
# выделим город и KartId в отдельную таблицу
info_city = info_dtp_clean[['KartId','city']]
info_city

,KartId,city
0,161237026,Екатеринбург
1,161340310,Екатеринбург
2,161238027,Екатеринбург
3,171517613,Екатеринбург
4,160935023,Екатеринбург
...,...,...
20219,225219802,Владивосток
20220,225222817,Владивосток
20221,225228096,Владивосток
20222,225219810,Владивосток


In [26]:
info_city['KartId'].duplicated().sum()

np.int64(0)

In [27]:
info_city = info_city.merge(cities_coords[['city', 'population']], on='city', how = 'left')
info_city['city'].unique()

array(['Екатеринбург', 'Владивосток'], dtype=object)

In [28]:
# добавим название города в первую таблицу с основными данными
main_clean = pd.merge(main_clean,info_city, on='KartId', how ='left')
main_clean.head()    

,KartId,date,time,district,dtp_type,dead,injured,ts_quantity,people_quantity,year,month,day,city,population
0,157114292,2015-01-02,12:30,Верх-Исетский,Наезд на стоящее ТС,0,1,2,3,2015,1,2,Екатеринбург,1544376.0
1,157122648,2015-01-02,17:15,Кировский,Наезд на пешехода,0,2,1,3,2015,1,2,Екатеринбург,1544376.0
2,157156173,2015-01-01,22:00,Октябрьский,Наезд на препятствие,0,1,1,1,2015,1,1,Екатеринбург,1544376.0
3,157223255,2015-01-01,02:20,Первомайский,Столкновение,0,2,2,3,2015,1,1,Владивосток,603519.0
4,157225233,2015-01-01,14:30,Первореченский,Наезд на пешехода,0,1,1,2,2015,1,1,Владивосток,603519.0


In [29]:
main_clean = main_clean.dropna(subset = ['city'])
main_clean.isna().sum() 

KartId             0
date               0
time               0
district           0
dtp_type           0
dead               0
injured            0
ts_quantity        0
people_quantity    0
year               0
month              0
day                0
city               0
population         0
dtype: int64

#### 1.3 Информация о транспортных средствах

In [30]:
try:
    query_ts = '''
        SELECT *                         
        FROM ts_df;
      '''
    ts = pd.read_sql_query(query_ts, con=engine)
except: 
    ts = pd.read_csv('ts_df.csv')

In [31]:
ts.info()
ts.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32517 entries, 0 to 32516
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   index     32517 non-null  int64  
 1   KartId    32517 non-null  int64  
 2   n_ts      32517 non-null  int64  
 3   ts_s      32517 non-null  object 
 4   t_ts      32316 non-null  object 
 5   marka_ts  30251 non-null  object 
 6   m_ts      30251 non-null  object 
 7   color     30941 non-null  object 
 8   r_rul     30374 non-null  object 
 9   g_v       30431 non-null  float64
 10  m_pov     6229 non-null   object 
 11  t_n       32517 non-null  object 
 12  f_sob     31092 non-null  object 
 13  o_pf      28090 non-null  object 
dtypes: float64(1), int64(3), object(10)
memory usage: 3.5+ MB


,index,KartId,n_ts,ts_s,t_ts,marka_ts,m_ts,color,r_rul,g_v,m_pov,t_n,f_sob,o_pf
0,0,161237026,1,Нет,"D-класс (средний) до 4,6 м",ВАЗ,Прочие модели ВАЗ,Синий,Передний,2013.0,Передний правый угол | Передний левый угол,Технические неисправности отсутствуют,Коллективная собственность (общественных объед...,NaN
1,1,161237026,2,Нет,"D-класс (средний) до 4,6 м",OPEL,Meriva,Серый,Передний,2007.0,Задний правый угол | Задний левый угол,Технические неисправности отсутствуют,Частная собственность,NaN
2,2,161340310,1,Да,"D-класс (средний) до 4,6 м",FORD,Focus,Черный,Передний,2007.0,NaN,Технические неисправности отсутствуют,Частная собственность,NaN
3,3,161238027,1,Нет,Минивэны и универсалы повышенной вместимости,MAZDA,CX-7,Иные цвета,Полноприводный,2010.0,Крыша | Полная деформация кузова | Смещение дв...,Технические неисправности отсутствуют,Частная собственность,NaN
4,4,171517613,1,Нет,"С-класс (малый средний, компактный) до 4,3 м",ВАЗ,Kalina,Фиолетовый,Передний,2008.0,Передний правый угол,Технические неисправности отсутствуют,Частная собственность,NaN


In [32]:
ts_clean = ts.drop(columns = ['index','ts_s',
                                'color','r_rul','f_sob','o_pf'])

ts_clean = ts_clean.rename(columns={'n_ts':'number_ts','t_ts': 'type_ts', 'm_ts': 'model_ts',
                            'g_v': 'year_release', 'm_pov': 'damage',
                            't_n': 'technical_defects'
                            })
#ts_clean['year_release'] = ts_clean['year_release'].astype(int)
clean_column_names(ts_clean)
ts_clean.head()

,KartId,number_ts,type_ts,marka_ts,model_ts,year_release,damage,technical_defects
0,161237026,1,"D-класс (средний) до 4,6 м",ВАЗ,Прочие модели ВАЗ,2013.0,Передний правый угол | Передний левый угол,Технические неисправности отсутствуют
1,161237026,2,"D-класс (средний) до 4,6 м",OPEL,Meriva,2007.0,Задний правый угол | Задний левый угол,Технические неисправности отсутствуют
2,161340310,1,"D-класс (средний) до 4,6 м",FORD,Focus,2007.0,NaN,Технические неисправности отсутствуют
3,161238027,1,Минивэны и универсалы повышенной вместимости,MAZDA,CX-7,2010.0,Крыша | Полная деформация кузова | Смещение дв...,Технические неисправности отсутствуют
4,171517613,1,"С-класс (малый средний, компактный) до 4,3 м",ВАЗ,Kalina,2008.0,Передний правый угол,Технические неисправности отсутствуют


In [33]:
# проверка уникальности карточки с номером
ts_clean['KartId'].duplicated().sum()

np.int64(12293)

#### 1.4 Информация об участниках ДТП

In [34]:
try: 
    query_uch = '''
        SELECT *
        FROM uch_data_df;
        '''
    uch = pd.read_sql_query(query_uch, con=engine)
except: 
    uch = pd.read_csv('uch_data_df.csv')

In [35]:
uch.info()
uch.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40748 entries, 0 to 40747
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            40748 non-null  int64  
 1   KartId           40748 non-null  int64  
 2   n_ts             40748 non-null  int64  
 3   K_UCH            40748 non-null  object 
 4   S_T              40522 non-null  object 
 5   POL              40630 non-null  object 
 6   V_ST             30132 non-null  float64
 7   ALCO             11354 non-null  float64
 8   SAFETY_BELT      40748 non-null  object 
 9   S_SM             40748 non-null  object 
 10  N_UCH            40748 non-null  int64  
 11  S_SEAT_GROUP     278 non-null    object 
 12  INJURED_CARD_ID  141 non-null    float64
 13  NPDD             40748 non-null  object 
 14  SOP_NPDD         40748 non-null  object 
dtypes: float64(3), int64(4), object(8)
memory usage: 4.7+ MB


,index,KartId,n_ts,K_UCH,S_T,POL,V_ST,ALCO,SAFETY_BELT,S_SM,N_UCH,S_SEAT_GROUP,INJURED_CARD_ID,NPDD,SOP_NPDD
0,0,161237026,1,Водитель,Не пострадал,Мужской,17.0,NaN,Да,Нет (не скрывался),1,NaN,NaN,Несоответствие скорости конкретным условиям дв...,Нет нарушений
1,1,161237026,2,Водитель,"Раненый, находящийся (находившийся) на стацион...",Женский,7.0,NaN,Да,Нет (не скрывался),2,NaN,NaN,Нет нарушений,Нет нарушений
2,2,161340310,1,Водитель,Не пострадал,Мужской,10.0,NaN,Да,"Скрылся, впоследствии разыскан (установлен)",1,NaN,NaN,"Несоблюдение условий, разрешающих движение тра...",Оставление места ДТП
3,3,161238027,1,Водитель,"Раненый, находящийся (находившийся) на стацион...",Женский,2.0,15.0,Да,Нет (не скрывался),1,NaN,NaN,Другие нарушения ПДД водителем,Управление ТС в состоянии алкогольного опьянения
4,4,171517613,1,Водитель,Не пострадал,Мужской,2.0,NaN,Да,Нет (не скрывался),2,NaN,NaN,Нет нарушений,Нет нарушений


In [36]:
uch_clean = uch.drop(columns = ['index','n_ts',
                                'N_UCH','S_SEAT_GROUP','INJURED_CARD_ID'])


uch_clean = uch_clean.rename(columns={'K_UCH': 'category_part', 'S_T': 'injury_type',
                            'V_ST': 'years_driving', 'ALCO': 'alcohol',
                            'SAFETY_BELT': 'safety_belt','POL':'pol'
                            })
clean_column_names(uch_clean)
uch_clean.head()

,KartId,category_part,injury_type,pol,years_driving,alcohol,safety_belt,S_SM,NPDD,SOP_NPDD
0,161237026,Водитель,Не пострадал,Мужской,17.0,NaN,Да,Нет (не скрывался),Несоответствие скорости конкретным условиям дв...,Нет нарушений
1,161237026,Водитель,"Раненый, находящийся (находившийся) на стацион...",Женский,7.0,NaN,Да,Нет (не скрывался),Нет нарушений,Нет нарушений
2,161340310,Водитель,Не пострадал,Мужской,10.0,NaN,Да,"Скрылся, впоследствии разыскан (установлен)","Несоблюдение условий, разрешающих движение тра...",Оставление места ДТП
3,161238027,Водитель,"Раненый, находящийся (находившийся) на стацион...",Женский,2.0,15.0,Да,Нет (не скрывался),Другие нарушения ПДД водителем,Управление ТС в состоянии алкогольного опьянения
4,171517613,Водитель,Не пострадал,Мужской,2.0,NaN,Да,Нет (не скрывался),Нет нарушений,Нет нарушений


In [37]:
# проверка уникальности карточки с номером
uch_clean['KartId'].duplicated().sum()

np.int64(20526)

#### 1.5 Выгрузка данных о погоде 


**Екатеринбург**

In [38]:
try:
    query_meteo_Ekb = '''
         SELECT *
         FROM екатеринбург_weather
         '''
    meteo_Ekb = pd.read_sql_query(query_meteo_Ekb, con=engine)
except:    
    meteo_Ekb = pd.read_csv('екатеринбург_weather.csv')

In [39]:
meteo_Ekb.info()
meteo_Ekb.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4019 entries, 0 to 4018
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   city                 4019 non-null   object 
 1   latitude             4019 non-null   float64
 2   longitude            4019 non-null   float64
 3   date                 4019 non-null   object 
 4   temperature_mean     4019 non-null   float64
 5   temperature_min      4019 non-null   float64
 6   temperature_max      4019 non-null   float64
 7   wind_speed_max       4019 non-null   float64
 8   precipitation_sum    4019 non-null   float64
 9   rain_sum             4019 non-null   float64
 10  snowfall_sum         4019 non-null   float64
 11  precipitation_hours  4019 non-null   float64
dtypes: float64(10), object(2)
memory usage: 376.9+ KB


,city,latitude,longitude,date,temperature_mean,temperature_min,temperature_max,wind_speed_max,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours
0,Екатеринбург,56.906853,60.63158,2015-01-01,-23.4,-27.7,-20.0,11.5,0.0,0.0,0.00,0.0
1,Екатеринбург,56.906853,60.63158,2015-01-02,-27.5,-29.7,-24.2,9.2,0.0,0.0,0.00,0.0
2,Екатеринбург,56.906853,60.63158,2015-01-03,-21.3,-29.8,-10.0,18.4,0.9,0.0,1.05,6.0
3,Екатеринбург,56.906853,60.63158,2015-01-04,-4.5,-8.1,-3.0,19.1,1.0,0.0,0.77,5.0
4,Екатеринбург,56.906853,60.63158,2015-01-05,-4.0,-6.4,-2.0,18.7,0.0,0.0,0.00,0.0


In [40]:
# удалим лишние столбцы
meteo_Ekb_clean = meteo_Ekb.drop(columns = ['latitude','longitude','precipitation_hours'])
                                        
#переименуем столбцы
meteo_Ekb_clean = meteo_Ekb_clean.rename(columns={
                            
                            'temperature_mean': 'temp_mean',
                            'temperature_min': 'temp_min', 
                            'temperature_max': 'temp_max',
                            'wind_speed_max': 'wind_max ', 
                            })

# мнеяем формат даты
meteo_Ekb_clean['date'] = pd.to_datetime(meteo_Ekb_clean['date'])


# добавляем отдельные столбы с годов, месяцем, днем
meteo_Ekb_clean['year'] = meteo_Ekb_clean['date'].dt.year
meteo_Ekb_clean['month'] = meteo_Ekb_clean['date'].dt.month
meteo_Ekb_clean['day'] = meteo_Ekb_clean['date'].dt.day

clean_column_names(meteo_Ekb_clean)

,city,date,temp_mean,temp_min,temp_max,wind_max,precipitation_sum,rain_sum,snowfall_sum,year,month,day
0,Екатеринбург,2015-01-01,-23.4,-27.7,-20.0,11.5,0.0,0.0,0.00,2015,1,1
1,Екатеринбург,2015-01-02,-27.5,-29.7,-24.2,9.2,0.0,0.0,0.00,2015,1,2
2,Екатеринбург,2015-01-03,-21.3,-29.8,-10.0,18.4,0.9,0.0,1.05,2015,1,3
3,Екатеринбург,2015-01-04,-4.5,-8.1,-3.0,19.1,1.0,0.0,0.77,2015,1,4
4,Екатеринбург,2015-01-05,-4.0,-6.4,-2.0,18.7,0.0,0.0,0.00,2015,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...
4014,Екатеринбург,2025-12-28,-10.1,-13.6,-7.5,15.2,1.6,0.0,1.12,2025,12,28
4015,Екатеринбург,2025-12-29,-11.5,-13.6,-8.9,12.6,3.1,0.0,2.24,2025,12,29
4016,Екатеринбург,2025-12-30,-14.6,-16.3,-12.4,10.2,0.2,0.0,0.14,2025,12,30
4017,Екатеринбург,2025-12-31,-13.6,-16.6,-11.4,18.3,0.1,0.0,0.07,2025,12,31


In [41]:
meteo_Ekb_clean.info()
meteo_Ekb_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4019 entries, 0 to 4018
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   city               4019 non-null   object        
 1   date               4019 non-null   datetime64[ns]
 2   temp_mean          4019 non-null   float64       
 3   temp_min           4019 non-null   float64       
 4   temp_max           4019 non-null   float64       
 5   wind_max           4019 non-null   float64       
 6   precipitation_sum  4019 non-null   float64       
 7   rain_sum           4019 non-null   float64       
 8   snowfall_sum       4019 non-null   float64       
 9   year               4019 non-null   int32         
 10  month              4019 non-null   int32         
 11  day                4019 non-null   int32         
dtypes: datetime64[ns](1), float64(7), int32(3), object(1)
memory usage: 329.8+ KB


,city,date,temp_mean,temp_min,temp_max,wind_max,precipitation_sum,rain_sum,snowfall_sum,year,month,day
0,Екатеринбург,2015-01-01,-23.4,-27.7,-20.0,11.5,0.0,0.0,0.00,2015,1,1
1,Екатеринбург,2015-01-02,-27.5,-29.7,-24.2,9.2,0.0,0.0,0.00,2015,1,2
2,Екатеринбург,2015-01-03,-21.3,-29.8,-10.0,18.4,0.9,0.0,1.05,2015,1,3
3,Екатеринбург,2015-01-04,-4.5,-8.1,-3.0,19.1,1.0,0.0,0.77,2015,1,4
4,Екатеринбург,2015-01-05,-4.0,-6.4,-2.0,18.7,0.0,0.0,0.00,2015,1,5


**Владивосток**

In [42]:
try:
    query_meteo_Vlad = '''
         SELECT *
         FROM владивосток_weather
         '''
    meteo_Vlad = pd.read_sql_query(query_meteo_Vlad, con=engine)
except:    
    meteo_Vlad = pd.read_csv('владивосток_weather.csv')

In [43]:
meteo_Vlad.info()
meteo_Vlad.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4019 entries, 0 to 4018
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   city                 4019 non-null   object 
 1   latitude             4019 non-null   float64
 2   longitude            4019 non-null   float64
 3   date                 4019 non-null   object 
 4   temperature_mean     4019 non-null   float64
 5   temperature_min      4019 non-null   float64
 6   temperature_max      4019 non-null   float64
 7   wind_speed_max       4019 non-null   float64
 8   precipitation_sum    4019 non-null   float64
 9   rain_sum             4019 non-null   float64
 10  snowfall_sum         4019 non-null   float64
 11  precipitation_hours  4019 non-null   float64
dtypes: float64(10), object(2)
memory usage: 376.9+ KB


,city,latitude,longitude,date,temperature_mean,temperature_min,temperature_max,wind_speed_max,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours
0,Владивосток,43.128292,131.98212,2015-01-01,-10.7,-12.3,-9.0,30.7,0.0,0.0,0.00,0.0
1,Владивосток,43.128292,131.98212,2015-01-02,-10.7,-12.8,-7.6,22.4,0.0,0.0,0.00,0.0
2,Владивосток,43.128292,131.98212,2015-01-03,-8.3,-11.0,-3.8,18.3,0.0,0.0,0.00,0.0
3,Владивосток,43.128292,131.98212,2015-01-04,-4.9,-9.7,-2.1,10.9,0.0,0.0,0.00,0.0
4,Владивосток,43.128292,131.98212,2015-01-05,-2.8,-10.0,0.9,21.2,1.7,1.0,0.49,6.0


In [44]:
# удалим лишние столбцы
meteo_Vlad_clean = meteo_Vlad.drop(columns = ['latitude','longitude','precipitation_hours'])
                                        
#переименуем столбцы
meteo_Vlad_clean = meteo_Vlad_clean.rename(columns={
                            
                            'temperature_mean': 'temp_mean',
                            'temperature_min': 'temp_min', 
                            'temperature_max': 'temp_max',
                            'wind_speed_max': 'wind_max ', 
                            })

# мнеяем формат даты
meteo_Vlad_clean['date'] = pd.to_datetime(meteo_Vlad_clean['date'])


# добавляем отдельные столбы с годов, месяцем, днем
meteo_Vlad_clean['year'] = meteo_Vlad_clean['date'].dt.year
meteo_Vlad_clean['month'] = meteo_Vlad_clean['date'].dt.month
meteo_Vlad_clean['day'] = meteo_Vlad_clean['date'].dt.day

clean_column_names(meteo_Vlad_clean)

,city,date,temp_mean,temp_min,temp_max,wind_max,precipitation_sum,rain_sum,snowfall_sum,year,month,day
0,Владивосток,2015-01-01,-10.7,-12.3,-9.0,30.7,0.0,0.0,0.00,2015,1,1
1,Владивосток,2015-01-02,-10.7,-12.8,-7.6,22.4,0.0,0.0,0.00,2015,1,2
2,Владивосток,2015-01-03,-8.3,-11.0,-3.8,18.3,0.0,0.0,0.00,2015,1,3
3,Владивосток,2015-01-04,-4.9,-9.7,-2.1,10.9,0.0,0.0,0.00,2015,1,4
4,Владивосток,2015-01-05,-2.8,-10.0,0.9,21.2,1.7,1.0,0.49,2015,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...
4014,Владивосток,2025-12-28,-6.0,-11.4,-1.0,17.4,0.0,0.0,0.00,2025,12,28
4015,Владивосток,2025-12-29,-1.7,-4.9,0.5,23.0,0.0,0.0,0.00,2025,12,29
4016,Владивосток,2025-12-30,-7.4,-9.6,-5.6,26.7,0.0,0.0,0.00,2025,12,30
4017,Владивосток,2025-12-31,-12.5,-14.0,-10.3,29.8,0.0,0.0,0.00,2025,12,31


In [45]:
meteo_Vlad_clean.info()
meteo_Vlad_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4019 entries, 0 to 4018
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   city               4019 non-null   object        
 1   date               4019 non-null   datetime64[ns]
 2   temp_mean          4019 non-null   float64       
 3   temp_min           4019 non-null   float64       
 4   temp_max           4019 non-null   float64       
 5   wind_max           4019 non-null   float64       
 6   precipitation_sum  4019 non-null   float64       
 7   rain_sum           4019 non-null   float64       
 8   snowfall_sum       4019 non-null   float64       
 9   year               4019 non-null   int32         
 10  month              4019 non-null   int32         
 11  day                4019 non-null   int32         
dtypes: datetime64[ns](1), float64(7), int32(3), object(1)
memory usage: 329.8+ KB


,city,date,temp_mean,temp_min,temp_max,wind_max,precipitation_sum,rain_sum,snowfall_sum,year,month,day
0,Владивосток,2015-01-01,-10.7,-12.3,-9.0,30.7,0.0,0.0,0.00,2015,1,1
1,Владивосток,2015-01-02,-10.7,-12.8,-7.6,22.4,0.0,0.0,0.00,2015,1,2
2,Владивосток,2015-01-03,-8.3,-11.0,-3.8,18.3,0.0,0.0,0.00,2015,1,3
3,Владивосток,2015-01-04,-4.9,-9.7,-2.1,10.9,0.0,0.0,0.00,2015,1,4
4,Владивосток,2015-01-05,-2.8,-10.0,0.9,21.2,1.7,1.0,0.49,2015,1,5


### 2. Загрузка очищенных данных в БД/CSV

In [46]:
def df_to_csv(df: pd.DataFrame, filename: str):
    ''' Функция загружает таблицы в csv файлы, создает папку для хранения.
        Аргументы: датафрейм и название папки\файла '''
        
    filepath = filename if filename.endswith('.csv') else f"{filename}.csv"
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    df.to_csv(f'{filename}.csv',
          sep=',',                   
          encoding='utf-8',          
          header=True,                
          decimal='.',     
          date_format='%Y-%m-%d',
          index=False) 

In [47]:
# загружаем очищенные датафреймы обратно в БД или csv (при недоступности БД)
try:
    main_clean.to_sql('main_clean', con=engine, if_exists='replace')
    info_dtp_clean.to_sql('info_dtp_clean', con=engine, if_exists='replace')
    ts_clean.to_sql('ts_clean', con=engine, if_exists='replace')
    uch_clean.to_sql('uch_clean', con=engine, if_exists='replace')
    calendar.to_sql('calendar', con=engine, if_exists='replace')
    info_city.to_sql('info_city', con=engine, if_exists='replace')
    meteo_Vlad_clean.to_sql('meteo_Vlad_clean', con=engine, if_exists='replace')
    meteo_Ekb_clean.to_sql('meteo_Ekb_clean', con=engine, if_exists='replace')
    print(f'Загрузка  успешна')
   
except Exception as e:
    print(f'Ошибка: {e}')
    df_to_csv(main_clean, 'data_for_dashboard\\main_clean')
    df_to_csv(info_dtp_clean, 'data_for_dashboard\\info_dtp_clean')
    df_to_csv(ts_clean, 'data_for_dashboard\\ts_clean')
    df_to_csv(uch_clean, 'data_for_dashboard\\uch_clean')
    df_to_csv(calendar, 'data_for_dashboard\\calendar')
    df_to_csv(info_city, 'data_for_dashboard\\info_city')
    df_to_csv(meteo_Vlad_clean, 'data_for_dashboard\\meteo_Vlad_clean')
    df_to_csv(meteo_Ekb_clean, 'data_for_dashboard\\meteo_Ekb_clean')
    print(f'Датафреймы загружены в csv')

Ошибка: (psycopg2.OperationalError) server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

[SQL: show standard_conforming_strings]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Датафреймы загружены в csv
